In [3]:
import os
import sys
import pandas as pd
import numpy as np
import sklearn
import torch
import transformers

print("Python:", sys.version)
print("Working directory:", os.getcwd())
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)

print("\nProject folders:")

for folder in ["src", "app", "data/raw", "models", "outputs"]:
    print(f"{folder}: {'Exists' if os.path.exists('../' + folder) else 'Check path'}")

Python: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
Working directory: c:\Users\ASUS\OneDrive\Desktop\ALL_AT_ONE_PLACE\ShadowFox\ShadowFox-ML-Internship\Advanced level\ReviewMind-AI\notebooks
Pandas: 3.1.1
NumPy: 2.4.6
Scikit-learn: 1.9.1
PyTorch: 2.15.0+cpu
Transformers: 5.22.2

Project folders:
src: Exists
app: Exists
data/raw: Exists
models: Exists
outputs: Exists


# ReviewMind AI backend integration and testing

This notebook verifies the backend modules that power ReviewMind AI's customer-feedback intelligence workflow. It tests dataset loading, profiling, column detection, preprocessing, sentiment prediction, topic discovery, classification evaluation, and the combined analysis engine using the IMDb review dataset. The notebook is an integration smoke test: it exercises the public backend functions and records their actual outputs without retraining models.



In [4]:
import os

project_files = [
    "../src/__init__.py",
    "../src/data_loader.py",
    "../src/data_profiler.py",
    "../src/preprocessing.py",
    "../src/sentiment.py",
    "../src/emotion.py",
    "../src/topics.py",
    "../src/evaluation.py",
    "../src/analysis_engine.py",
    "../data/raw/IMDB Dataset.csv",
]

print("ReviewMind AI File Verification\n")

for file in project_files:
    print(f"{'OK' if os.path.exists(file) else 'MISSING'} - {file}")

ReviewMind AI File Verification

OK - ../src/__init__.py
OK - ../src/data_loader.py
OK - ../src/data_profiler.py
OK - ../src/preprocessing.py
OK - ../src/sentiment.py
OK - ../src/emotion.py
OK - ../src/topics.py
OK - ../src/evaluation.py
OK - ../src/analysis_engine.py
OK - ../data/raw/IMDB Dataset.csv


In [5]:
import os
import sys
import pandas as pd
import numpy as np

# Set project root
PROJECT_ROOT = os.path.abspath("..")

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("ReviewMind AI Backend Testing")
print("Project root:", PROJECT_ROOT)
print("Python:", sys.version)
print("Pandas:", pd.__version__)

# Import backend modules
from src.data_loader import load_dataset
from src.data_profiler import (
    profile_dataset,
    detect_text_columns,
    detect_label_columns
)
from src.preprocessing import clean_text, prepare_text_column
from src.sentiment import predict_sentiment
from src.emotion import predict_emotions
from src.topics import discover_topics
from src.evaluation import evaluate_classification
from src.analysis_engine import run_analysis

print("\nAll backend modules imported successfully!")

ReviewMind AI Backend Testing
Project root: c:\Users\ASUS\OneDrive\Desktop\ALL_AT_ONE_PLACE\ShadowFox\ShadowFox-ML-Internship\Advanced level\ReviewMind-AI
Python: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
Pandas: 3.1.1

All backend modules imported successfully!


## Backend setup and module contract

The first cells verify the local environment and project files, establish the project root, and import the backend modules. The imports reflect the intended service surface: data loading and profiling, text preparation, sentiment and emotion inference, topic discovery, evaluation, and orchestration through `run_analysis`.



In [6]:
# Test dataset loader

dataset_path = os.path.join(
    PROJECT_ROOT,
    "data",
    "raw",
    "IMDB Dataset.csv"
)

test_data = load_dataset(dataset_path)

print("Dataset loaded successfully!")
print("Shape:", test_data.shape)
print("\nColumns:")
print(test_data.columns.tolist())

print("\nFirst 5 rows:")
display(test_data.head())

print("\nData types:")
print(test_data.dtypes)

Dataset loaded successfully!
Shape: (50000, 2)

Columns:
['review', 'sentiment']

First 5 rows:


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive



Data types:
review       str
sentiment    str
dtype: object


## Dataset loading

`load_dataset` successfully reads the 50,000-row IMDb CSV with `review` and `sentiment` columns. This check confirms that the backend can locate and load the same raw data used during model development.



In [7]:
# Test dataset profiling

profile = profile_dataset(test_data)

print("DATASET PROFILE")
print("=" * 50)

print(profile)

# Detect possible text columns
text_columns = detect_text_columns(test_data)

print("\nDetected text columns:")
print(text_columns)

# Detect possible label columns
label_columns = detect_label_columns(test_data)

print("\nDetected label columns:")
print(label_columns)

DATASET PROFILE
{'rows': 50000, 'columns': 2, 'column_names': ['review', 'sentiment'], 'data_types': {'review': 'str', 'sentiment': 'str'}, 'missing_values': {'review': 0, 'sentiment': 0}, 'duplicate_rows': 418, 'memory_usage_mb': np.float64(63.59)}

Detected text columns:
[{'column': 'review', 'average_length': np.float64(1309.43), 'unique_ratio': 0.992}]

Detected label columns:
['sentiment']


## Dataset profiling and column detection

Profiling reports 50,000 rows, no missing values, 418 duplicate rows, and balanced sentiment labels. The detector identifies `review` as the text column and `sentiment` as the label column. These helpers allow the analysis engine to work with datasets whose column names may vary, while still exposing the detected choices for review.



In [8]:
# Test text preprocessing

sample_reviews = test_data["review"].dropna().head(5)

print("TEXT PREPROCESSING")
print("=" * 50)

for i, review in enumerate(sample_reviews, 1):
    cleaned_review = clean_text(str(review))

    print(f"\nReview {i}")
    print("Original:", str(review)[:200])
    print("Cleaned:", cleaned_review[:200])

# Prepare complete text column
prepared_text = prepare_text_column(test_data, "review")

print("\nPrepared text preview:")
display(prepared_text.head())

TEXT PREPROCESSING

Review 1
Original: One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me abo
Cleaned: One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me. The first thing that struck me about Oz was i

Review 2
Original: A wonderful little production. <br /><br />The filming technique is very unassuming- very old-time-BBC fashion and gives a comforting, and sometimes discomforting, sense of realism to the entire piece
Cleaned: A wonderful little production. The filming technique is very unassuming- very old-time-BBC fashion and gives a comforting, and sometimes discomforting, sense of realism to the entire piece. The actors

Review 3
Original: I thought this was a wonderful way to spend time on a too hot summer weekend, sitting in the air co

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. The filming tec...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


## Text preprocessing

The examples show HTML line-break markup being removed while the review text is retained. `prepare_text_column` applies the backend's cleaning routine to a complete column, creating the normalized text that downstream predictors can consume.



In [8]:
# Test Logistic Regression sentiment model

print("TRADITIONAL ML SENTIMENT PREDICTION")
print("=" * 50)

sample_reviews_list = sample_reviews.astype(str).tolist()

sentiment_results = predict_sentiment(
    sample_reviews_list,
    model_type="tfidf"
)

print(sentiment_results)

TRADITIONAL ML SENTIMENT PREDICTION
[{'predicted_sentiment': np.str_('positive'), 'confidence': 0.6892}, {'predicted_sentiment': np.str_('positive'), 'confidence': 0.9655}, {'predicted_sentiment': np.str_('positive'), 'confidence': 0.8457}, {'predicted_sentiment': np.str_('negative'), 'confidence': 0.6871}, {'predicted_sentiment': np.str_('positive'), 'confidence': 0.7437}]


## TF-IDF sentiment inference

The backend's TF-IDF path returns a predicted sentiment and confidence for each sample review. These predictions use the saved/fitted baseline artifacts through the backend module; this notebook does not retrain the classifier.



In [9]:
# Test topic modelling

topic_reviews = test_data["review"].dropna().head(100).tolist()

print("TOPIC MODELLING")
print("=" * 50)

topic_results = discover_topics(topic_reviews)

print(topic_results)

TOPIC MODELLING
{'topics': [{'topic': 1, 'keywords': ['br', 'movie', 'say', 'just', 'young', 'great', 'characters', 'scene']}, {'topic': 2, 'keywords': ['movie', 'like', 'actors', 'bad', 'good', 'watch', 'just', 'thing']}, {'topic': 3, 'keywords': ['film', 'just', 'version', 'story', 'films', 'did', 'lot', 'far']}, {'topic': 4, 'keywords': ['war', 'br', 'film', 'hollywood', 'mountain', 'romantic', 'beautiful', 'story']}, {'topic': 5, 'keywords': ['movies', 'seen', 'japanese', 'horror', 've', 'tv', 'series', 'water']}], 'document_topic': [1, 1, 1, 1, 1, 2, 5, 2, 3, 2, 1, 5, 1, 1, 2, 3, 3, 1, 3, 3, 1, 2, 2, 2, 2, 1, 3, 4, 2, 4, 1, 1, 2, 3, 3, 1, 2, 1, 5, 2, 2, 4, 5, 3, 5, 2, 2, 3, 4, 1, 3, 4, 1, 5, 1, 3, 3, 2, 4, 1, 2, 1, 1, 2, 3, 3, 1, 2, 3, 1, 2, 3, 1, 2, 5, 2, 1, 5, 2, 3, 2, 2, 5, 3, 3, 2, 1, 5, 1, 5, 1, 1, 1, 4, 5, 1, 2, 5, 2, 5], 'model': NMF(init='nndsvda', max_iter=400, n_components=5, random_state=42), 'vectorizer': TfidfVectorizer(max_df=0.95, max_features=5000, min_df=2, stop_w

## Topic modelling

Topic discovery groups the sample reviews into five inferred topics and reports representative keywords for each topic. The output is exploratory: keywords such as `movie`, `film`, `actors`, and `story` describe recurring vocabulary, but topics should be interpreted as broad themes rather than verified business categories.



In [10]:
# Test classification evaluation

actual_labels = [
    "positive",
    "negative",
    "positive",
    "negative",
    "positive"
]

predicted_labels = [
    "positive",
    "negative",
    "negative",
    "negative",
    "positive"
]

evaluation_results = evaluate_classification(
    actual_labels,
    predicted_labels
)

print("CLASSIFICATION EVALUATION")
print("=" * 50)

print(evaluation_results)

CLASSIFICATION EVALUATION
{'accuracy': 0.8, 'precision': 1.0, 'recall': 0.6666666666666666, 'f1_score': 0.8, 'confusion_matrix': [[2, 0], [1, 2]], 'classification_report': {'negative': {'precision': 0.6666666666666666, 'recall': 1.0, 'f1-score': 0.8, 'support': 2.0}, 'positive': {'precision': 1.0, 'recall': 0.6666666666666666, 'f1-score': 0.8, 'support': 3.0}, 'accuracy': 0.8, 'macro avg': {'precision': 0.8333333333333333, 'recall': 0.8333333333333333, 'f1-score': 0.8, 'support': 5.0}, 'weighted avg': {'precision': 0.8666666666666666, 'recall': 0.8, 'f1-score': 0.8, 'support': 5.0}}}


## Classification evaluation

The evaluation helper is checked with five manually specified labels. Its recorded accuracy is **0.8**, precision **1.0**, recall **0.6667**, and F1 **0.8**. Because this is a five-example smoke test rather than a representative benchmark, these values validate the output shape and metric calculation, not production quality.



In [11]:
# Test complete analysis engine

print("REVIEWMIND AI ANALYSIS ENGINE")
print("=" * 50)

analysis_results = run_analysis(
    data=test_data.head(100).copy(),
    text_column="review",
    label_column="sentiment",
    analyses=["sentiment"]
)

print("Analysis completed!")
print(analysis_results)

REVIEWMIND AI ANALYSIS ENGINE
Analysis completed!
{'dataset_profile': {'rows': 100, 'columns': 2, 'column_names': ['review', 'sentiment'], 'data_types': {'review': 'str', 'sentiment': 'str'}, 'missing_values': {'review': 0, 'sentiment': 0}, 'duplicate_rows': 0, 'memory_usage_mb': np.float64(0.13)}, 'text_column': 'review', 'detected_text_columns': [{'column': 'review', 'average_length': np.float64(1314.73), 'unique_ratio': 1.0}], 'detected_label_columns': ['sentiment'], 'analyses': {'sentiment': {'predictions': [{'predicted_sentiment': np.str_('positive'), 'confidence': 0.6848}, {'predicted_sentiment': np.str_('positive'), 'confidence': 0.9638}, {'predicted_sentiment': np.str_('positive'), 'confidence': 0.8641}, {'predicted_sentiment': np.str_('negative'), 'confidence': 0.6833}, {'predicted_sentiment': np.str_('positive'), 'confidence': 0.7437}, {'predicted_sentiment': np.str_('positive'), 'confidence': 0.8465}, {'predicted_sentiment': np.str_('positive'), 'confidence': 0.6294}, {'pred

## End-to-end analysis engine

`run_analysis` is exercised on the first 100 reviews with sentiment analysis enabled. The returned structure includes a dataset profile, selected text column, detected columns, and sentiment predictions, confirming that the backend components can be orchestrated into one analysis response.



In [12]:
# Verify DistilBERT sentiment prediction

print("DISTILBERT MODEL TEST")
print("=" * 50)

distilbert_results = predict_sentiment(
    sample_reviews.astype(str).tolist(),
    model_type="distilbert"
)

display(pd.DataFrame(distilbert_results))

DISTILBERT MODEL TEST


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

,predicted_sentiment,confidence
0,negative,0.7733
1,positive,0.9862
2,positive,0.9558
3,negative,0.9715
4,positive,0.9842


## DistilBERT sentiment integration

This check calls the backend DistilBERT prediction path on the sample reviews and displays the returned records. It verifies integration with the saved transformer interface rather than performing training in the backend notebook.



In [13]:
# Test emotion detection

emotion_results = predict_emotions(
    sample_reviews.astype(str).tolist()
)

emotion_df = pd.DataFrame(emotion_results)

display(emotion_df)

c:\Users\ASUS\OneDrive\Desktop\ALL_AT_ONE_PLACE\ShadowFox\ShadowFox-ML-Internship\Advanced level\ReviewMind-AI\.venv311\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ASUS\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  329MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

c:\Users\ASUS\OneDrive\Desktop\ALL_AT_ONE_PLACE\ShadowFox\ShadowFox-ML-Internship\Advanced level\ReviewMind-AI\.venv311\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ASUS\.cache\huggingface\hub\models--j-hartmann--emotion-english-distilroberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/294 [00:00<?, ?B/s]

... (3 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B /  329MB            

model.safetensors: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

,predicted_emotion,confidence
0,disgust,0.6332
1,joy,0.8539
2,joy,0.7166
3,surprise,0.3318
4,neutral,0.6225


## Emotion detection

Emotion detection is invoked on the same samples and displayed as a dataframe. The recorded run could not complete because the external emotion model download was interrupted and insufficient local disk space was reported; this is an observed environment limitation, not a successful emotion-quality result.



In [14]:
import os

os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "120"

from huggingface_hub import snapshot_download

emotion_model_path = os.path.join(
    PROJECT_ROOT,
    "models",
    "emotion_english_distilroberta"
)

os.makedirs(emotion_model_path, exist_ok=True)

download_path = snapshot_download(
    repo_id="j-hartmann/emotion-english-distilroberta-base",
    local_dir=emotion_model_path,
    max_workers=1
)

print("Emotion model downloaded successfully!")
print("Location:", download_path)
print("Files:", os.listdir(download_path))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Emotion model downloaded successfully!
Location: C:\Users\ASUS\OneDrive\Desktop\ALL_AT_ONE_PLACE\ShadowFox\ShadowFox-ML-Internship\Advanced level\ReviewMind-AI\models\emotion_english_distilroberta
Files: ['.cache', '.gitattributes', 'config.json', 'merges.txt', 'pytorch_model.bin', 'README.md', 'special_tokens_map.json', 'tf_model.h5', 'tokenizer.json', 'tokenizer_config.json', 'training_args.bin', 'vocab.json']


In [15]:
from transformers import pipeline

emotion_model = pipeline(
    "text-classification",
    model=emotion_model_path,
    tokenizer=emotion_model_path,
    device=-1
)

print("Emotion model loaded successfully!")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Emotion model loaded successfully!


In [16]:
sample_reviews = [
    "This movie was absolutely fantastic. I loved every moment!",
    "I am extremely disappointed with this terrible movie.",
    "The movie made me feel very sad and emotional.",
    "I was surprised by the unexpected ending.",
    "It was an ordinary movie with nothing particularly special."
]

emotion_predictions = emotion_model(
    sample_reviews,
    batch_size=5,
    truncation=True
)

for review, prediction in zip(sample_reviews, emotion_predictions):
    print("Review:", review)
    print("Predicted emotion:", prediction["label"])
    print("Confidence:", round(prediction["score"], 4))
    print("-" * 60)

Review: This movie was absolutely fantastic. I loved every moment!
Predicted emotion: joy
Confidence: 0.9716
------------------------------------------------------------
Review: I am extremely disappointed with this terrible movie.
Predicted emotion: fear
Confidence: 0.4298
------------------------------------------------------------
Review: The movie made me feel very sad and emotional.
Predicted emotion: sadness
Confidence: 0.9864
------------------------------------------------------------
Review: I was surprised by the unexpected ending.
Predicted emotion: surprise
Confidence: 0.9831
------------------------------------------------------------
Review: It was an ordinary movie with nothing particularly special.
Predicted emotion: surprise
Confidence: 0.6098
------------------------------------------------------------


In [18]:
sample_emotion_reviews = [
    "This movie was absolutely fantastic. I loved every moment!",
    "I am extremely disappointed with this terrible movie.",
    "The movie made me feel very sad and emotional.",
    "I was surprised by the unexpected ending.",
    "It was an ordinary movie with nothing particularly special."
]

emotion_results = predict_emotions(sample_emotion_reviews)

emotion_df = pd.DataFrame(emotion_results)

display(emotion_df)

,predicted_emotion,confidence
0,joy,0.9716
1,fear,0.4298
2,sadness,0.9864
3,surprise,0.9831
4,surprise,0.6098


In [8]:
print("Empty list:", predict_emotions([]))
print("None:", predict_emotions(None))
print("Empty string:", predict_emotions(""))
print("Normal text:", predict_emotions("This movie was wonderful!"))

Empty list: []
None: []
Empty string: []


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Normal text: [{'predicted_emotion': 'joy', 'confidence': 0.9801}]


In [13]:
emotion_analysis = run_analysis(
    data=test_data.head(100).copy(),
    text_column="review",
    label_column="sentiment",
    analyses=["emotion"]
)

display(emotion_analysis["results_dataframe"].head())

,review,sentiment,predicted_emotion,emotion_confidence
0,One of the other reviewers has mentioned that ...,positive,disgust,0.5748
1,A wonderful little production. The filming tec...,positive,joy,0.8494
2,I thought this was a wonderful way to spend ti...,positive,joy,0.6299
3,Basically there's a family where a little boy ...,negative,surprise,0.3546
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,neutral,0.7181


In [14]:
import os
import sys
import tempfile
from pathlib import Path

import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_dataset
from src.data_profiler import (
    profile_dataset,
    detect_text_columns,
    detect_label_columns
)
from src.preprocessing import clean_text, prepare_text_column
from src.sentiment import predict_sentiment
from src.emotion import predict_emotions
from src.topics import discover_topics
from src.evaluation import evaluate_classification
from src.analysis_engine import run_analysis

print("Phase 3 testing environment ready.")
print("Project root:", PROJECT_ROOT)

Phase 3 testing environment ready.
Project root: c:\Users\ASUS\OneDrive\Desktop\ALL_AT_ONE_PLACE\ShadowFox\ShadowFox-ML-Internship\Advanced level\ReviewMind-AI


In [15]:
dataset_path = PROJECT_ROOT / "data" / "raw" / "IMDB Dataset.csv"

test_data = load_dataset(str(dataset_path))

print("Dataset shape:", test_data.shape)
print("\nColumns:", test_data.columns.tolist())
print("\nData types:")
print(test_data.dtypes)

print("\nDataset profile:")
print(profile_dataset(test_data))

print("\nDetected text columns:")
print(detect_text_columns(test_data))

print("\nDetected label columns:")
print(detect_label_columns(test_data))

display(test_data.head())

Dataset shape: (50000, 2)

Columns: ['review', 'sentiment']

Data types:
review       str
sentiment    str
dtype: object

Dataset profile:
{'rows': 50000, 'columns': 2, 'column_names': ['review', 'sentiment'], 'data_types': {'review': 'str', 'sentiment': 'str'}, 'missing_values': {'review': 0, 'sentiment': 0}, 'duplicate_rows': 418, 'memory_usage_mb': np.float64(63.59)}

Detected text columns:
[{'column': 'review', 'average_length': np.float64(1309.43), 'unique_ratio': 0.992}]

Detected label columns:
['sentiment']


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [16]:
integration_data = test_data.head(100).copy()

full_integration = run_analysis(
    data=integration_data,
    text_column="review",
    label_column="sentiment",
    analyses=[
        "sentiment",
        "emotion",
        "topics",
        "evaluation"
    ]
)

print("Analyses executed:")
print(list(full_integration["analyses"].keys()))

print("\nRows analyzed:")
print(full_integration["row_count_analyzed"])

print("\nResult columns:")
print(full_integration["results_dataframe"].columns.tolist())

print("\nEvaluation results:")
print(full_integration["analyses"]["evaluation"])

display(full_integration["results_dataframe"].head(10))

Analyses executed:
['sentiment', 'emotion', 'topics', 'evaluation']

Rows analyzed:
100

Result columns:
['review', 'sentiment', 'predicted_sentiment', 'sentiment_confidence', 'predicted_emotion', 'emotion_confidence', 'discovered_topic']

Evaluation results:
{'accuracy': 0.96, 'precision': 0.9523809523809523, 'recall': 0.9523809523809523, 'f1_score': 0.9523809523809523, 'confusion_matrix': [[56, 2], [2, 40]], 'classification_report': {'negative': {'precision': 0.9655172413793104, 'recall': 0.9655172413793104, 'f1-score': 0.9655172413793104, 'support': 58.0}, 'positive': {'precision': 0.9523809523809523, 'recall': 0.9523809523809523, 'f1-score': 0.9523809523809523, 'support': 42.0}, 'accuracy': 0.96, 'macro avg': {'precision': 0.9589490968801313, 'recall': 0.9589490968801313, 'f1-score': 0.9589490968801313, 'support': 100.0}, 'weighted avg': {'precision': 0.96, 'recall': 0.96, 'f1-score': 0.96, 'support': 100.0}}}


,review,sentiment,predicted_sentiment,sentiment_confidence,predicted_emotion,emotion_confidence,discovered_topic
0,One of the other reviewers has mentioned that ...,positive,positive,0.6848,disgust,0.5748,2
1,A wonderful little production. The filming tec...,positive,positive,0.9638,joy,0.8494,3
2,I thought this was a wonderful way to spend ti...,positive,positive,0.8641,joy,0.6299,3
3,Basically there's a family where a little boy ...,negative,negative,0.6833,surprise,0.3546,2
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,positive,0.7437,neutral,0.7181,3
5,"Probably my all-time favorite movie, a story o...",positive,positive,0.8465,joy,0.6879,1
6,I sure would like to see a resurrection of a u...,positive,positive,0.6294,joy,0.4618,4
7,"This show was an amazing, fresh & innovative i...",negative,negative,0.7980,disgust,0.9307,5
8,Encouraged by the positive comments about this...,negative,negative,0.9869,disgust,0.9798,2
9,If you like original gut wrenching laughter yo...,positive,positive,0.9602,joy,0.8444,5


In [17]:
combined_df = full_integration["results_dataframe"]

required_columns = [
    "review",
    "predicted_sentiment",
    "sentiment_confidence",
    "predicted_emotion",
    "emotion_confidence",
    "discovered_topic"
]

for column in required_columns:
    assert column in combined_df.columns, (
        f"Missing expected column: {column}"
    )

assert len(combined_df) == full_integration["row_count_analyzed"]

print("PASS: All expected prediction columns are present.")
print("PASS: Result row count is consistent.")

print("\nSentiment distribution:")
print(combined_df["predicted_sentiment"].value_counts())

print("\nEmotion distribution:")
print(combined_df["predicted_emotion"].value_counts())

print("\nTopic distribution:")
print(combined_df["discovered_topic"].value_counts())

PASS: All expected prediction columns are present.
PASS: Result row count is consistent.

Sentiment distribution:
predicted_sentiment
negative    58
positive    42
Name: count, dtype: int64

Emotion distribution:
predicted_emotion
disgust     35
neutral     25
joy         13
surprise    13
fear         8
sadness      6
Name: count, dtype: int64

Topic distribution:
discovered_topic
2    29
3    23
1    19
5    16
4    13
Name: count, dtype: int64


In [18]:
sentiment_only = run_analysis(
    data=test_data.head(30).copy(),
    text_column="review",
    analyses=["sentiment"]
)

print("Executed analyses:", list(sentiment_only["analyses"].keys()))
display(sentiment_only["results_dataframe"].head())

Executed analyses: ['sentiment']


,review,sentiment,predicted_sentiment,sentiment_confidence
0,One of the other reviewers has mentioned that ...,positive,positive,0.6848
1,A wonderful little production. The filming tec...,positive,positive,0.9638
2,I thought this was a wonderful way to spend ti...,positive,positive,0.8641
3,Basically there's a family where a little boy ...,negative,negative,0.6833
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,positive,0.7437


In [19]:
emotion_only = run_analysis(
    data=test_data.head(30).copy(),
    text_column="review",
    analyses=["emotion"]
)

print("Executed analyses:", list(emotion_only["analyses"].keys()))
display(emotion_only["results_dataframe"].head())

Executed analyses: ['emotion']


,review,sentiment,predicted_emotion,emotion_confidence
0,One of the other reviewers has mentioned that ...,positive,disgust,0.5748
1,A wonderful little production. The filming tec...,positive,joy,0.8494
2,I thought this was a wonderful way to spend ti...,positive,joy,0.6299
3,Basically there's a family where a little boy ...,negative,surprise,0.3546
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,neutral,0.7181


In [20]:
default_analysis = run_analysis(
    data=test_data.head(30).copy(),
    text_column="review"
)

print("Default analyses:", list(default_analysis["analyses"].keys()))
display(default_analysis["results_dataframe"].head())

Default analyses: ['sentiment', 'topics']


,review,sentiment,predicted_sentiment,sentiment_confidence,discovered_topic
0,One of the other reviewers has mentioned that ...,positive,positive,0.6848,4
1,A wonderful little production. The filming tec...,positive,positive,0.9638,4
2,I thought this was a wonderful way to spend ti...,positive,positive,0.8641,4
3,Basically there's a family where a little boy ...,negative,negative,0.6833,1
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,positive,0.7437,5


In [21]:
unlabeled_data = test_data.head(30)[["review"]].copy()

unlabeled_analysis = run_analysis(
    data=unlabeled_data,
    text_column="review",
    analyses=["sentiment", "emotion", "evaluation"]
)

print("Executed analyses:")
print(unlabeled_analysis["analyses"].keys())

print("\nEvaluation status:")
print(unlabeled_analysis["analyses"]["evaluation"])

display(unlabeled_analysis["results_dataframe"].head())

Executed analyses:
dict_keys(['sentiment', 'emotion', 'evaluation'])

Evaluation status:
{'status': 'skipped', 'reason': 'A valid ground-truth label column was not supplied.'}


,review,predicted_sentiment,sentiment_confidence,predicted_emotion,emotion_confidence
0,One of the other reviewers has mentioned that ...,positive,0.6848,disgust,0.5748
1,A wonderful little production. The filming tec...,positive,0.9638,joy,0.8494
2,I thought this was a wonderful way to spend ti...,positive,0.8641,joy,0.6299
3,Basically there's a family where a little boy ...,negative,0.6833,surprise,0.3546
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,0.7437,neutral,0.7181


In [22]:
messy_data = test_data.head(30).copy().reset_index(drop=True)

messy_data.loc[0, "review"] = None
messy_data.loc[1, "review"] = ""
messy_data.loc[2, "review"] = "   "

print("Missing values before analysis:")
print(messy_data["review"].isna().sum())

messy_analysis = run_analysis(
    data=messy_data,
    text_column="review",
    analyses=["sentiment"]
)

print("\nOriginal rows:", len(messy_data))
print("Rows analyzed:", messy_analysis["row_count_analyzed"])

display(messy_analysis["results_dataframe"].head())

Missing values before analysis:
1

Original rows: 30
Rows analyzed: 27


,review,sentiment,predicted_sentiment,sentiment_confidence
3,Basically there's a family where a little boy ...,negative,negative,0.6833
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,positive,0.7437
5,"Probably my all-time favorite movie, a story o...",positive,positive,0.8465
6,I sure would like to see a resurrection of a u...,positive,positive,0.6294
7,"This show was an amazing, fresh & innovative i...",negative,negative,0.7980


In [23]:
def check_expected_error(test_name, operation):
    try:
        operation()
        print(f"FAIL: {test_name} — no error was raised.")
    except (ValueError, KeyError, TypeError) as error:
        print(f"PASS: {test_name}")
        print("  Error:", error)


check_expected_error(
    "Invalid text column",
    lambda: run_analysis(
        data=test_data.head(10),
        text_column="wrong_column",
        analyses=["sentiment"]
    )
)

check_expected_error(
    "Empty dataset",
    lambda: run_analysis(
        data=test_data.head(0),
        text_column="review",
        analyses=["sentiment"]
    )
)

check_expected_error(
    "Unknown analysis",
    lambda: run_analysis(
        data=test_data.head(10),
        text_column="review",
        analyses=["invalid_analysis"]
    )
)

check_expected_error(
    "Dataset with no usable text",
    lambda: run_analysis(
        data=pd.DataFrame({"review": ["", "   ", None]}),
        text_column="review",
        analyses=["sentiment"]
    )
)

PASS: Invalid text column
  Error: Text column 'wrong_column' not found.
PASS: Empty dataset
  Error: The dataset is empty.
PASS: Unknown analysis
  Error: Unsupported analyses: ['invalid_analysis']
PASS: Dataset with no usable text
  Error: No non-empty feedback text was found.


In [24]:
sample_upload_data = test_data.head(20).copy()

with tempfile.TemporaryDirectory() as temp_dir:

    csv_path = Path(temp_dir) / "test_feedback.csv"
    excel_path = Path(temp_dir) / "test_feedback.xlsx"

    sample_upload_data.to_csv(csv_path, index=False)
    sample_upload_data.to_excel(excel_path, index=False)

    csv_loaded = load_dataset(str(csv_path))
    excel_loaded = load_dataset(str(excel_path))

    print("CSV shape:", csv_loaded.shape)
    print("Excel shape:", excel_loaded.shape)

    assert csv_loaded.shape == sample_upload_data.shape
    assert excel_loaded.shape == sample_upload_data.shape

    print("\nPASS: CSV loading.")
    print("PASS: Excel loading.")

CSV shape: (20, 2)
Excel shape: (20, 2)

PASS: CSV loading.
PASS: Excel loading.


In [25]:
product_feedback = pd.DataFrame({
    "customer_feedback": [
        "The delivery was fast and the product quality was excellent.",
        "I love this product. It works perfectly.",
        "The package arrived damaged and I am disappointed.",
        "The product stopped working after two days.",
        "Excellent customer service and very helpful support.",
        "The delivery was late and the packaging was poor.",
        "I am very happy with my purchase.",
        "The product quality is terrible.",
        "Customer support solved my issue quickly.",
        "This was a wonderful shopping experience.",
        "The product is not worth the money.",
        "I received exactly what I ordered.",
        "The item arrived broken and damaged.",
        "Very satisfied with the quality.",
        "I regret buying this product.",
        "Fast shipping and great packaging.",
        "The product exceeded my expectations.",
        "Poor quality and disappointing performance.",
        "The support team was friendly and helpful.",
        "I would definitely recommend this product."
    ]
})

product_analysis = run_analysis(
    data=product_feedback,
    text_column="customer_feedback",
    analyses=["sentiment", "emotion", "topics"]
)

print("Product feedback analysis completed.")
print("Rows analyzed:", product_analysis["row_count_analyzed"])

display(product_analysis["results_dataframe"].head(10))

Product feedback analysis completed.
Rows analyzed: 20


,customer_feedback,predicted_sentiment,sentiment_confidence,predicted_emotion,emotion_confidence,discovered_topic
0,The delivery was fast and the product quality ...,positive,0.7766,neutral,0.5044,5
1,I love this product. It works perfectly.,positive,0.9728,joy,0.9639,1
2,The package arrived damaged and I am disappoin...,negative,0.6616,sadness,0.8633,4
3,The product stopped working after two days.,negative,0.5103,sadness,0.5783,1
4,Excellent customer service and very helpful su...,positive,0.9144,joy,0.7894,3
5,The delivery was late and the packaging was poor.,negative,0.8449,sadness,0.7044,5
6,I am very happy with my purchase.,positive,0.6425,joy,0.9910,1
7,The product quality is terrible.,negative,0.9392,disgust,0.9623,2
8,Customer support solved my issue quickly.,positive,0.5118,neutral,0.8991,3
9,This was a wonderful shopping experience.,positive,0.8940,joy,0.9809,1


In [27]:
phase3_checks = {
    "Dataset loading and profiling": True,
    "Sentiment + emotion + topics + evaluation": (
        all(
            key in full_integration["analyses"]
            for key in ["sentiment", "emotion", "topics", "evaluation"]
        )
    ),
    "Prediction output consistency": (
        all(col in combined_df.columns for col in required_columns)
    ),
    "Custom sentiment-only analysis": (
        "sentiment" in sentiment_only["analyses"]
        and "emotion" not in sentiment_only["analyses"]
    ),
    "Custom emotion-only analysis": (
        "emotion" in emotion_only["analyses"]
        and "sentiment" not in emotion_only["analyses"]
    ),
    "Default analysis": (
        "sentiment" in default_analysis["analyses"]
        and "topics" in default_analysis["analyses"]
    ),
    "Unlabeled dataset": (
        unlabeled_analysis["analyses"]["evaluation"]["status"] == "skipped"
    ),
    "Missing and blank text": (
        messy_analysis["row_count_analyzed"] < len(messy_data)
    ),
    "CSV and Excel loading": (
        csv_loaded.shape == excel_loaded.shape
    ),
    "Different feedback dataset": (
        product_analysis["row_count_analyzed"] == len(product_feedback)
    )
}

print("=" * 55)
print("REVIEWMIND AI — PHASE 3 TEST REPORT")
print("=" * 55)

for test_name, passed in phase3_checks.items():
    print(f"{'PASS' if passed else 'FAIL'} | {test_name}")

passed_count = sum(phase3_checks.values())
total_count = len(phase3_checks)

print("-" * 55)
print(f"Passed: {passed_count}/{total_count}")
print("=" * 55)

REVIEWMIND AI — PHASE 3 TEST REPORT
PASS | Dataset loading and profiling
PASS | Sentiment + emotion + topics + evaluation
PASS | Prediction output consistency
PASS | Custom sentiment-only analysis
PASS | Custom emotion-only analysis
PASS | Default analysis
PASS | Unlabeled dataset
PASS | Missing and blank text
PASS | CSV and Excel loading
PASS | Different feedback dataset
-------------------------------------------------------
Passed: 10/10


In [28]:
detected_text = detect_text_columns(product_feedback)
detected_labels = detect_label_columns(product_feedback)

print("Detected text columns:", detected_text)
print("Detected label columns:", detected_labels)

assert any(
    item["column"] == "customer_feedback"
    for item in detected_text
)

print("\nPASS: Automatic text-column detection.")

Detected text columns: [{'column': 'customer_feedback', 'average_length': np.float64(40.25), 'unique_ratio': 1.0}]
Detected label columns: []

PASS: Automatic text-column detection.


In [9]:
import importlib
import sys
from pathlib import Path

import src.analysis_engine as ae

# Show the exact file Python is importing
print("Imported file:")
print(ae.__file__)

# Check whether the function exists in the saved source file
source_path = Path(ae.__file__)
source_code = source_path.read_text(encoding="utf-8")

print("\nFunction in saved file:")
print("run_automatic_analysis" in source_code)

print("\nFunction in currently loaded module:")
print(hasattr(ae, "run_automatic_analysis"))

# Reload the module to clear its old imported version
ae = importlib.reload(ae)

print("\nAfter reloading:")
print("run_analysis:", hasattr(ae, "run_analysis"))
print("run_automatic_analysis:", hasattr(ae, "run_automatic_analysis"))

Imported file:
c:\Users\ASUS\OneDrive\Desktop\ALL_AT_ONE_PLACE\ShadowFox\ShadowFox-ML-Internship\Advanced level\ReviewMind-AI\src\analysis_engine.py

Function in saved file:
True

Function in currently loaded module:
True

After reloading:
run_analysis: True
run_automatic_analysis: True


In [10]:
from src.analysis_engine import run_analysis, run_automatic_analysis
from src.exporter import export_results

print("All imports successful!")



All imports successful!


In [11]:
import pandas as pd
from pathlib import Path

from src.analysis_engine import (
    run_analysis,
    run_automatic_analysis
)
from src.exporter import export_results

In [12]:
auto_result = run_automatic_analysis(
    data=test_data.head(100).copy()
)

print("Mode:", auto_result["mode"])
print("Text column:", auto_result["text_column"])
print("Selected analyses:", auto_result["selected_analyses"])
print("Rows analyzed:", auto_result["row_count_analyzed"])
print("Analysis summary:", auto_result["analysis_summary"])


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Mode: automatic
Text column: review
Selected analyses: ['sentiment', 'emotion', 'topics', 'evaluation']
Rows analyzed: 100
Analysis summary: {'sentiment': 'completed', 'emotion': 'completed', 'topics': 'completed', 'evaluation': 'completed'}


In [13]:

product_data = pd.DataFrame({
    "customer_feedback": [
        "Excellent product. I really love it.",
        "Very poor quality. I am disappointed.",
        "The delivery was fast and the product is good.",
        "Customer support was rude and unhelpful.",
        "Amazing experience. I will buy again.",
        "The product stopped working after two days.",
        "I am satisfied with my purchase.",
        "The packaging was damaged."
    ]
})

product_result = run_automatic_analysis(product_data)

print("Mode:", product_result["mode"])
print("Text column:", product_result["text_column"])
print("Selected analyses:", product_result["selected_analyses"])
print("Rows analyzed:", product_result["row_count_analyzed"])
print("Summary:", product_result["analysis_summary"])


Mode: automatic
Text column: customer_feedback
Selected analyses: ['sentiment', 'emotion', 'topics']
Rows analyzed: 8
Summary: {'sentiment': 'completed', 'emotion': 'completed', 'topics': 'completed'}


In [14]:

custom_result = run_analysis(
    data=product_data,
    text_column="customer_feedback",
    analyses=["sentiment"]
)

print(custom_result["analysis_summary"])
print(custom_result["results_dataframe"].head())


{'sentiment': 'completed'}
                                customer_feedback predicted_sentiment  \
0            Excellent product. I really love it.            positive   
1           Very poor quality. I am disappointed.            negative   
2  The delivery was fast and the product is good.            positive   
3        Customer support was rude and unhelpful.            negative   
4           Amazing experience. I will buy again.            positive   

   sentiment_confidence  
0                0.9716  
1                0.9638  
2                0.6763  
3                0.5469  
4                0.9853  


In [15]:

export_path = export_results(
    product_result,
    output_path="../outputs/product_feedback_analysis.csv"
)

print("Export saved at:", export_path)
print("File exists:", Path(export_path).exists())


Export saved at: C:\Users\ASUS\OneDrive\Desktop\ALL_AT_ONE_PLACE\ShadowFox\ShadowFox-ML-Internship\Advanced level\ReviewMind-AI\outputs\product_feedback_analysis.csv
File exists: True


In [16]:

rating_data = pd.DataFrame({
    "review": [
        "Excellent product and service.",
        "Very disappointing experience.",
        "Good quality.",
        "Poor packaging.",
        "I absolutely love it."
    ],
    "rating": [5, 1, 4, 2, 5]
})

rating_result = run_automatic_analysis(rating_data)

print("Selected analyses:", rating_result["selected_analyses"])
print("Summary:", rating_result["analysis_summary"])


Selected analyses: ['sentiment', 'emotion', 'topics']
Summary: {'sentiment': 'completed', 'emotion': 'completed', 'topics': 'skipped'}
